# Recompute match_source: NASA Confirmed Check

**Logic**: For each `Target` in the CSV, query NASA's `pscomppars` (confirmed planets table). If the host star is found → `confirmed`, otherwise → `none`.

**Input**: `merge_results.CSV`

**Output**: A copy with updated `match_source` column.

In [ ]:
!pip install -q numpy pandas astropy astroquery

In [ ]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

df = pd.read_csv('merge_results.CSV')
print(f'Loaded: {len(df)} rows')
df.head(3)

In [ ]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

_CACHE = {}

def check_nasa_confirmed(target):
    '''Return True if the target host star is in NASA confirmed planets table.'''
    t = str(target)
    if t in _CACHE:
        return _CACHE[t]
    try:
        result = NasaExoplanetArchive.query_object(t)
        is_confirmed = result is not None and len(result) > 0
        _CACHE[t] = is_confirmed
        return is_confirmed
    except Exception:
        _CACHE[t] = False
        return False

sources = []
for i, row in df.iterrows():
    tgt = row['Target']
    ok = check_nasa_confirmed(tgt)
    label = 'confirmed' if ok else 'none'
    sources.append(label)
    print(f"[{i+1:>3}/{len(df)}] {str(tgt):<25} -> {label}")

df['match_source'] = sources

df.to_csv('merge_results.CSV', index=False, encoding='utf-8-sig')
print(f"\nDone. confirmed: {sources.count('confirmed')}  none: {sources.count('none')}")